<a href="https://colab.research.google.com/github/Presto5572/Fina-os/blob/main/neural_networks/basic_neural_networks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Basic Neural Networks

MIST 5400 Spring 2026
By: Pearl Yu

Creadit to the help from Aditya Deshpande and Chris Volinsky.

# In-class Exercise at the end

In [1]:
#Loading Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelBinarizer

# Additional libraries for modeling + evaluation
from sklearn import metrics
from sklearn.model_selection import train_test_split

#Read in the dataset

We're using the DirectMarketing dataset. Each record represents an individual who was targeted with a direct marketing offer.  The offer was a solicitation to make a charitable donation.


After downloading, we could open the folder at the left, and drag the downloaded local csv to the current working directory.


In [2]:
# This is cloning Pearl's Github repository that has the dataset.
!git clone https://github.com/pearl-yu/mist5400spring2026.git
%cd mist5400spring2026/neural_networks/

Cloning into 'mist5400spring2026'...
remote: Enumerating objects: 57, done.
remote: Counting objects: 100% (57/57), done.
remote: Compressing objects: 100% (53/53), done.
remote: Total 57 (delta 16), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (57/57), 1.59 MiB | 2.48 MiB/s, done.
Resolving deltas: 100% (16/16), done.
/content/mist5400spring2026/neural_networks


In [3]:
# read in the dataset
from google.colab import files
df = pd.read_csv("/content/DirectMarketing.csv")

Below is just some data cleaning procedures.

In [4]:
# remove cases where Firstdate == 0 using .loc
df = df.loc[df.Firstdate != 0]

# replace gavr and glast with log versions of same features using .loc
df_clean = df
df_clean['gavr'] = np.log(df.gavr+1)
df_clean['glast'] = np.log(df.glast+1)
income_cat = pd.Categorical(df['Income'], categories=[0,1,2,3,4,5,6,7])
df_clean['Income'] = income_cat

rfaf2_cat = pd.Categorical(df['rfaf2'], categories=[1,2,3,4])
df_clean['rfaf2'] = rfaf2_cat

df_clean = pd.get_dummies(df_clean, columns=['rfaa2', 'pepstrfl','Income','rfaf2'],drop_first=True)
df_clean.head()
# Create a new feature 'tenure'
df_clean['tenure'] = df_clean['Lastdate'] - df_clean['Firstdate']

# maybe check to see this is always greater than zero?
df_clean['tenure'].min()
today = df_clean['Lastdate'].max()
df_clean['recency'] = today - df_clean['Lastdate']

# remove Firstdate and Lastdate
df_clean = df_clean.drop(['Firstdate', 'Lastdate'], axis=1)


In [5]:
# Take a look at a few rows
df_clean.head()


,Amount,glast,gavr,class,rfaa2_E,rfaa2_F,rfaa2_G,pepstrfl_X,Income_1,Income_2,Income_3,Income_4,Income_5,Income_6,Income_7,rfaf2_2,rfaf2_3,rfaf2_4,tenure,recency
0,0.06,3.931826,3.433987,0,False,False,True,False,False,False,True,False,False,False,False,False,False,False,100,193
1,0.16,3.044522,3.070376,1,False,False,True,True,False,True,False,False,False,False,False,False,False,True,401,100
2,0.20,1.791759,2.277267,0,True,False,False,False,False,False,False,False,False,False,False,False,False,True,93,99
3,0.13,3.258097,3.157000,0,False,False,True,False,False,False,False,False,False,True,False,True,False,False,194,99
4,0.10,3.258097,2.602690,0,False,False,True,False,False,False,False,False,False,False,False,False,False,False,201,191


In [6]:
# Specify feature columns and target
X = df_clean.drop(['class'], axis=1)
y = df_clean['class']

# Split data into training and testing sets
random_state_value = 99
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=random_state_value
)

## Class imbalance check (why accuracy can be misleading)

If most records are non-donors, a model can get high accuracy by predicting "non-donor" for almost everyone.
We'll compute the **majority-class baseline accuracy** as a sanity check.

In [7]:
# Baseline: always predict the majority class
print("Class distribution:")
display(y.value_counts())

baseline_accuracy = y.value_counts(normalize=True).max()
print(f"Baseline accuracy (always predict majority class): {baseline_accuracy:.3f}")

Class distribution:


,count
class,
0,182059
1,9716


Baseline accuracy (always predict majority class): 0.949


## Neural Networks (using Keras)

In [8]:
#Loading Libraries

import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.preprocessing import StandardScaler

In [9]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

The four lines below define the neural network model.

`kmodel = Sequential()`: This initializes a Sequential model, which is a linear stack of layers. It's the simplest way to build a Keras model.

`kmodel.add(Dense(12,input_shape =(19,), activation = "relu"))`: This adds the first hidden layer to the model.
- It's a Dense layer, meaning each neuron in this layer is connected to every neuron in the previous layer.
- It has 12 neurons and expects input with 19 features (input_shape =(19,)). The activation = "relu"

`kmodel.add(Dense(8,activation = "relu"))`: This adds a second hidden Dense layer with 8 neurons, also using the ReLU activation function.

`kmodel.add(Dense(1,activation = "sigmoid"))`: This adds the output layer.
- It's a Dense layer with 1 neuron, and it uses a sigmoid activation function. - The sigmoid function is commonly used for binary classification problems (like this one, where the target class is 0 or 1), as it outputs a probability between 0 and 1.


In [10]:
# Defining SIMPLE Keras Model
kmodel = Sequential()
kmodel.add(Dense(12,input_shape =(19,), activation = "relu"))
kmodel.add(Dense(8,activation = "relu"))
kmodel.add(Dense(1,activation = "sigmoid"))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [11]:
kmodel.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 12)             │           240 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 8)              │           104 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │             9 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 353 (1.38 KB)

 Trainable params: 353 (1.38 KB)

 Non-trainable params: 0 (0.00 B)

Let's ask Gemini to explain the next line of code.

In [12]:
#Compile Keras Model
kmodel.compile(loss = "binary_crossentropy", optimizer = "adam")


In [13]:
#Fitting Keras Model
kmodel.fit(X_train_scaled,y_train,epochs = 20, batch_size = 256)

Epoch 1/20
600/600 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - loss: 0.4276
Epoch 2/20
600/600 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.1986
Epoch 3/20
600/600 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.1999
Epoch 4/20
600/600 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.1956
Epoch 5/20
600/600 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.1972
Epoch 6/20
600/600 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - loss: 0.1953
Epoch 7/20
600/600 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - loss: 0.1957
Epoch 8/20
600/600 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step - loss: 0.1970
Epoch 9/20
600/600 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.1931
Epoch 10/20
600/600 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.1970
Epoch 11/20
600/600 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.1924
Epoch 12/20
600/600 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.1933
Epoch 13/20
600/600 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.1940
Epoch 14/20
600/600 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.1963
Epoch 15/20
600/600 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - lo

In [14]:
from sklearn.metrics import accuracy_score

# Get predicted probabilities for the positive class (class 1) for the simple Keras model
y_prob_keras_simple_test = kmodel.predict(X_test_scaled).ravel()

# Calculate predictions using a 0.5 threshold
y_pred_keras_simple_test = (y_prob_keras_simple_test >= 0.5).astype(int)

# Calculate accuracy for the simple Keras model
accuracy_keras_simple = accuracy_score(y_test, y_pred_keras_simple_test)

print(f"Simple Keras Model - Testing Accuracy: {accuracy_keras_simple:.4f}")

1199/1199 ━━━━━━━━━━━━━━━━━━━━ 1s 757us/step
Simple Keras Model - Testing Accuracy: 0.9477


Wow the accuracy (using the default cutoff of 0.5) is high, but remember we discussed that accuracy can be a misleading metric, especially when there's imbalance of class in the dataset? Let's put that aside for now. We'll explore that in the assignment.  

# In-class exercise: Adding a dropout layer

In the cell below, can you add a dropout layer after one of the dense layers? You can ask Gemini to add it.

Then run through the rest of the cells, upload the completed notebook to eLc.

In [15]:
## Now we make it more complex, with an extra layer, and Dropout

kmodel2 = Sequential()
kmodel2.add(Dense(12,input_shape =(19,), activation = "relu")) # Change input_shape to (19,)
kmodel2.add(Dense(8,activation = "relu"))
kmodel2.add(Dense(1,activation = "sigmoid"))


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [16]:
kmodel.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 12)             │           240 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 8)              │           104 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │             9 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,061 (4.15 KB)

 Trainable params: 353 (1.38 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 708 (2.77 KB)

In [17]:
#Compile Keras Model
kmodel2.compile(loss = "binary_crossentropy", optimizer = "adam", metrics =['accuracy'])


In [18]:
#Fitting Keras Model
kmodel2.fit(X_train_scaled,y_train,epochs = 30, batch_size = 256)

Epoch 1/30
600/600 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.9059 - loss: 0.3397
Epoch 2/30
600/600 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9505 - loss: 0.1984
Epoch 3/30
600/600 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9493 - loss: 0.1986
Epoch 4/30
600/600 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9496 - loss: 0.1964
Epoch 5/30
600/600 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9499 - loss: 0.1957
Epoch 6/30
600/600 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9505 - loss: 0.1930
Epoch 7/30
600/600 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9499 - loss: 0.1946
Epoch 8/30
600/600 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9499 - loss: 0.1947
Epoch 9/30
600/600 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9488 - loss: 0.1983
Epoch 10/30
600/600 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9495 - loss: 0.1955
Epoch 11/30
600/600 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9499 - loss: 0.1944
Epoch 12/30
600/600 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step

In [19]:
# Get predicted probabilities for the positive class (class 1) for the complex Keras model
y_prob_keras_test = kmodel2.predict(X_test_scaled).ravel()

# Calculate predictions using a 0.5 threshold for the complex Keras model
y_pred_keras_complex_test = (y_prob_keras_test >= 0.5).astype(int)

# Calculate accuracy for the complex Keras model
accuracy_keras_complex = accuracy_score(y_test, y_pred_keras_complex_test)

print(f"More Complex Keras Model (with Dropout) - Testing Accuracy: {accuracy_keras_complex:.4f}")

1199/1199 ━━━━━━━━━━━━━━━━━━━━ 1s 647us/step
More Complex Keras Model (with Dropout) - Testing Accuracy: 0.9477
